In [ ]:
import torch
from lieflow.groups import SEplus12, P3
import matplotlib.pyplot as plt
%matplotlib widget

In [ ]:
seplus12 = SEplus12()

In [ ]:
def random_time_like(N, ε_stab=1e-4):
    vec = torch.randn(N, 3)
    vec[..., 0] = (1 + torch.randn(N)**2).sqrt().clamp_min(1 + ε_stab) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    # restrict such that |angle| < pi
    return vec * ((1 - 1000. * ε_stab) * torch.pi / inner_product(vec, vec, keepdim=True).sqrt()).clamp_max(1.)

def random_space_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = torch.clamp_min(1/(1 + torch.randn(N)**2).sqrt(), 0.1) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def random_light_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def inner_product(vec1, vec2, keepdim=False):
    return (vec1 @ seplus12.η * vec2).sum(-1, keepdim=keepdim)

In [ ]:
N = 2**14

In [ ]:
test_vec = random_time_like(N)
lightlike = inner_product(test_vec, test_vec) < 0.01
N_timelike = N - lightlike.sum()
translations = torch.randn(N_timelike, 3)
test_vec = test_vec[~lightlike]
lorentz = (test_vec[..., None, None] * seplus12.soplus12.lie_algebra_basis).sum(-3)
test_gen = torch.zeros(N_timelike, 4, 4)
seplus12.pack_translation_lorentz(test_gen, translations, lorentz);

In [ ]:
(seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().max()

In [ ]:
(seplus12.exp(seplus12.log(seplus12.exp(test_gen))) - seplus12.exp(test_gen)).abs().max()

In [ ]:
bad_set = (seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()

In [ ]:
test_vec = random_space_like(N)
lightlike = inner_product(test_vec, test_vec) > -0.01
N_spacelike = N - lightlike.sum()
translations = torch.randn(N_spacelike, 3)
test_vec = test_vec[~lightlike]
lorentz = (test_vec[..., None, None] * seplus12.soplus12.lie_algebra_basis).sum(-3)
test_gen = torch.zeros(N_spacelike, 4, 4)
seplus12.pack_translation_lorentz(test_gen, translations, lorentz);

In [ ]:
(seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().max()

In [ ]:
(seplus12.exp(seplus12.log(seplus12.exp(test_gen))) - seplus12.exp(test_gen)).abs().max()

In [ ]:
bad_set = (seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()

In [ ]:
test_vec = random_light_like(N)
lightlike = inner_product(test_vec, test_vec).abs() < 0.01
N_lightlike = lightlike.sum()
translations = torch.randn(N_lightlike, 3)
test_vec = test_vec[lightlike]
lorentz = (test_vec[..., None, None] * seplus12.soplus12.lie_algebra_basis).sum(-3)
test_gen = torch.zeros(N_lightlike, 4, 4)
seplus12.pack_translation_lorentz(test_gen, translations, lorentz);

In [ ]:
(seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().max()

In [ ]:
(seplus12.exp(seplus12.log(seplus12.exp(test_gen))) - seplus12.exp(test_gen)).abs().max()

In [ ]:
bad_set = (seplus12.log(seplus12.exp(test_gen)) - test_gen).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()

In [ ]:
p3 = P3()

In [ ]:
def random_unit_time_like(N):
    vec = torch.randn(N, 3)
    vec[..., 0] = (1 + vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

In [ ]:
x_0s = torch.randn(N, 3)
n_0s = random_unit_time_like(N)
p_0s = p3.pack_event_velocity(x_0s, n_0s)
x_1s = torch.randn(N, 3)
n_1s = random_unit_time_like(N)
p_1s = p3.pack_event_velocity(x_1s, n_1s)

In [ ]:
gens = p3.get_generator(p_0s, p_1s, generator="spacelike")

In [ ]:
(p3.act(p3.seplus12.exp(gens), p_0s) - p_1s).abs().max()

In [ ]:
(p3.act(p3.seplus12.exp(gens), p_0s) - p_1s).abs().max()
bad_set = (p3.act(p3.seplus12.exp(gens), p_0s) - p_1s).abs().amax(dim=(-2, -1)) > 0.01
(bad_set * 1.).mean()

In [ ]:
p3 = P3()

x_0 = torch.zeros(3)
n_0_t = torch.tensor([1., 0., 0.])
n_0_x = torch.tensor([0., 1., 0.])
n_0_y = torch.tensor([0., 0., 1.])
p_0_t = p3.pack_event_velocity(x_0, n_0_t)
p_0_x = p3.pack_event_velocity(x_0, n_0_x)
p_0_y = p3.pack_event_velocity(x_0, n_0_y)

def g_to_frame(g):
    return p3.act(g, p_0_t), p3.act(g, p_0_x), p3.act(g, p_0_y)

def lift_to_SEplus12(p):
    A = p3.get_generator(p_0_t, p)
    random_rotation = p3.seplus12.exp(2 * torch.pi * torch.rand(*p.shape[:-2], 1, 1) * p3.seplus12.lie_algebra_basis[-3])
    return p3.seplus12.L(p3.seplus12.exp(A), random_rotation)
    
def project_to_P3(g):
    return p3.act(g, p_0_t)

def plot_p(ax, p, **kwargs):
    x, n = p3.get_event_velocity(p)
    return ax.quiver(x[..., 0], x[..., 1], x[..., 2], n[..., 0], n[..., 1], n[..., 2], **kwargs)

In [ ]:
x = torch.randn(32, 3)
n = random_unit_time_like(32)
p = p3.pack_event_velocity(x, n)
g = lift_to_SEplus12(p)
p_t, p_x, p_y = g_to_frame(g)
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
plot_p(ax, p, length=0.1, color="k")
plot_p(ax, p_t, length=0.2, color="red")
plot_p(ax, p_x, length=0.2, color="green")
plot_p(ax, p_y, length=0.2, color="blue")
ax.set_aspect("equal")

In [ ]:
components = 2. * torch.randn(6)
A = (components[:, None, None] * p3.seplus12.lie_algebra_basis).sum(-3)
g = p3.seplus12.exp(A)
ts = torch.linspace(0., 1., 20)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_t, p_x_t, p_y_t = g_to_frame(g_t)
p_t, p_x, p_y = g_to_frame(g)
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
plot_p(ax, p_t_t, length=0.1, alpha=0.5, color="red")
plot_p(ax, p_x_t, length=0.1, alpha=0.5, color="green")
plot_p(ax, p_y_t, length=0.1, alpha=0.5, color="blue")
plot_p(ax, p_t, length=0.1, alpha=0.5, color="red")
plot_p(ax, p_x, length=0.1, alpha=0.5, color="green")
plot_p(ax, p_y, length=0.1, alpha=0.5, color="blue")
ax.set_xlabel("t")
ax.set_ylabel("x")
ax.set_zlabel("y")
ax.set_aspect("equal");

In [ ]:
x_1 = torch.tensor([0., 0., 0.])
n_1 = torch.tensor([1., 0., 0.])
p_1 = p3.pack_position_orientation(x_1, n_1)
x_2 = torch.randn(3)
n_2 = random_unit_time_like(1)[0]
p_2 = p3.pack_position_orientation(x_2, n_2)

ts = torch.linspace(0., 1., 20)
A = p3.get_generator(p_1, p_2, generator="spacelike")
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_space = p3.act(g_t, p_1)

A = p3.get_generator(p_1, p_2, generator="lightlike")
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_light = p3.act(g_t, p_1)

A = p3.get_generator(p_1, p_2, generator="timelike")
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_time = p3.act(g_t, p_1)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
plot_p(ax, p_t_space, length=0.1, alpha=0.5, color="blue", label="spacelike")
plot_p(ax, p_t_light, length=0.1, alpha=0.5, color="orange", label="lightlike")
plot_p(ax, p_t_time, length=0.1, alpha=0.5, color="green", label="timelike")
plot_p(ax, p_1, length=0.1, color="red")
plot_p(ax, p_2, length=0.1, color="red")
ax.set_aspect("equal")
ax.legend();

In [ ]:
x_1 = torch.tensor([0., 0., 0.])
n_1 = torch.tensor([1., 0., 0.])
p_1 = p3.pack_position_orientation(x_1, n_1)
x_2 = torch.randn(3)
n_2 = random_unit_time_like(1)[0]
p_2 = p3.pack_position_orientation(x_2, n_2)

ts = torch.linspace(0., 1., 20)

A = p3.get_generator(p_1, p_2, generator="spacelike")
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_space = p3.act(g_t, p_1)

φ_1 = 0.5
A = p3.get_generator(p_1, p_2, generator=φ_1)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_φ_1 = p3.act(g_t, p_1)

φ_2 = 1.
A = p3.get_generator(p_1, p_2, generator=φ_2)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_φ_2 = p3.act(g_t, p_1)

φ_3 = 1.5
A = p3.get_generator(p_1, p_2, generator=φ_3)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_φ_3 = p3.act(g_t, p_1)

φ_4 = 2.
A = p3.get_generator(p_1, p_2, generator=φ_4)
ts = torch.linspace(0., 1., 20)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_φ_4 = p3.act(g_t, p_1)

φ_5 = 2.5
A = p3.get_generator(p_1, p_2, generator=φ_5)
ts = torch.linspace(0., 1., 20)
g_t = p3.seplus12.exp(ts[..., None, None] * A)
p_t_φ_5 = p3.act(g_t, p_1)

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
plot_p(ax, p_t_space, length=0.1, alpha=0.5, color="red", label="spacelike")
plot_p(ax, p_t_φ_1, length=0.1, alpha=0.5, color="blue", label=f"φ = {φ_1:.2f}")
plot_p(ax, p_t_φ_2, length=0.1, alpha=0.5, color="green", label=f"φ = {φ_2:.2f}")
plot_p(ax, p_t_φ_3, length=0.1, alpha=0.5, color="orange", label=f"φ = {φ_3:.2f}")
plot_p(ax, p_t_φ_4, length=0.1, alpha=0.5, color="yellow", label=f"φ = {φ_4:.2f}")
plot_p(ax, p_t_φ_5, length=0.1, alpha=0.5, color="purple", label=f"φ = {φ_5:.2f}")
plot_p(ax, p_1, length=0.1, color="red")
plot_p(ax, p_2, length=0.1, color="red")
ax.set_aspect("equal")
ax.legend();